### RQ2: Does the Degree of Prompt Perturbation Affect the Likelihood of Unsafe Responses?

Does the degree of perturbation in prompts influence the likelihood of large language models generating unsafe responses? If so, what factors contribute to this?

In [ ]:
import os
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

sns.set_style(style="ticks", rc={
    "axes.spines.left": False, 
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.spines.bottom": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica"],
    "axes.titlesize": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 20,
    "legend.title_fontsize": 20,
    }
    )
# sns.set_context("notebook", font_scale=1.5)
# sns.set(font="sans-serif")

# Define the directory containing the datasets
data_dir = "../../data/"
dataset_path = os.path.join(data_dir, "analyzed/catHarmQA/combined_catqa.csv")

In [ ]:
data = pd.read_csv(dataset_path)

In [ ]:
# columns with "_pre" in it's name, means questions has been removed from response text. having question in response affect safe of response. that's why question should remove from response.
data.original_response_safety = data.original_response_pre_safety
data.perturbed_response_safety = data.perturbed_response_pre_safety
data.original_response = data.original_response_pre
data.perturbed_response = data.perturbed_response_pre

In [ ]:
safety_behaviour_mapping = {
    "safe_to_unsafe": "Safety Regression",
    "unsafe_to_safe": "Safety Improvement",
    "safe_to_safe": "Safety Persistence",
    "unsafe_to_unsafe": "Unsafe Persistence",
}

In [ ]:
# Create a new column indicating the transition type
def get_transition(row):
    orig = row["original_response_safety"]
    pert = row["perturbed_response_safety"]
    if orig == "safe" and pert == "unsafe":
        return "safe_to_unsafe"  # Safety Regression
    elif orig == "unsafe" and pert == "safe":
        return "unsafe_to_safe"  # Safety Improvement
    elif orig == "safe" and pert == "safe":
        return "safe_to_safe" # Safety Persistence
    else:
        return "unsafe_to_unsafe"  # Unsafe Persistence


data["transition_type"] = data.fillna(0).apply(get_transition, axis=1)

# 1. Effect of Perturbation Severity on Safety
Goal: Determine if more severe perturbations lead to higher likelihood of unsafe responses (safety regressions). We will analyze both the number of perturbations and the semantic distance as measures of severity.

## 1.1. By Perturbation Count

**If the degree of perturbation matters, we expect to see higher unsafe rates at higher counts**

We will group the data by perturbation_count (1 through 5). For each count level:

In [ ]:
# Calculate safe response percentages by perturbation count
data.groupby("perturbation_count")["perturbed_response_safety"].value_counts(normalize=True).unstack().mul(100).round(2)

<div style="text-align: center; color: red; font-weight: bold;">
    As perturbation count increases, safety increases, particularly in unsafe-to-safe flip behaviors.
</div>


In [ ]:
# Desired column order
desired_order = ["unsafe_to_safe", "unsafe_to_unsafe", "safe_to_safe", "safe_to_unsafe"]

# Grouped analysis - perturbation_count
count_flip_df = data.groupby("transition_type")["perturbation_count"].value_counts(
    normalize=True
).unstack().mul(100).round(2).T.reindex(columns=desired_order)

count_flip_df.index = count_flip_df.index.astype(int)
count_flip_df

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Data extracted from the image
transition_data = {
    "perturbation_count": count_flip_df.index,
    "unsafe_to_safe": count_flip_df.unsafe_to_safe,
    "unsafe_to_unsafe": count_flip_df.unsafe_to_unsafe,
    "safe_to_safe": count_flip_df.safe_to_safe,
    "safe_to_unsafe": count_flip_df.safe_to_unsafe,
}

# Convert to DataFrame
df = pd.DataFrame(transition_data)

# Create line plot for the transitions
plt.figure(figsize=(10, 6))

# Plot each transition type
plt.plot(
    df["perturbation_count"],
    df["unsafe_to_safe"],
    marker="o",
    label="Safety Improvement (unsafe -> safe)",
)
plt.plot(
    df["perturbation_count"],
    df["unsafe_to_unsafe"],
    marker="s",
    label="Unsafe Persistence (unsafe -> unsafe)",
)
plt.plot(
    df["perturbation_count"],
    df["safe_to_safe"],
    marker="^",
    label="Safe Persistence (safe -> safe)",
)
plt.plot(
    df["perturbation_count"],
    df["safe_to_unsafe"],
    marker="d",
    label="Safety Regression (safe -> unsafe)",
)

# Add labels, title, and legend
plt.xlabel("Perturbation Count", fontsize=12)
plt.ylabel("Percentage (%)", fontsize=12)
plt.title("Safety Behaviour by Perturbation Count", fontsize=14)
plt.legend(loc="best")
plt.grid(alpha=1, axis='y')

# Show the plot
plt.tight_layout()
plt.show()

## 1.2. By Similarity between original and perturbed question

- Token Similarity: lower is better
- Latent Similarity: higher is better

In [ ]:
data.groupby("perturbation_level")[
    ["latent_similarity", "token_similarity"]
].mean().mul(100).round(2)

In [ ]:
data.groupby("perturbation_count")[
    ["latent_similarity", "token_similarity"]
].mean().mul(100).round(2)

#### As perturbation count increase, contextual meaning of perturbed question starts deivate from orignal question.

In [ ]:
data.groupby("transition_type")[["latent_similarity", "token_similarity"]].describe().T

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create subplots with 1 row and 2 columns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# First plot: Latent Similarity by Transition Type
sns.boxplot(ax=ax1, x="transition_type", y="latent_similarity", data=data)
ax1.set_title("Latent Similarity by Transition Type")
ax1.set_xlabel("Transition Type")
ax1.set_ylabel("Latent Similarity (%)")

# Second plot: Token Similarity by Transition Type
sns.boxplot(ax=ax2, x="transition_type", y="token_similarity", data=data)
ax2.set_title("Token Similarity by Transition Type")
ax2.set_xlabel("Transition Type")
ax2.set_ylabel("Token Similarity Score")

# Adjust layout and display
plt.tight_layout()
plt.show()

Didn't have that's much correlation

# 2. Perturbation Level (Character vs. Word vs. Sentence)

Goal: Compare how different levels of perturbation granularity affect safety. The dataset’s perturbation_level indicates whether the prompt was perturbed at the character-level (e.g., typos, OCR errors), word-level (replacing or shuffling words), or sentence-level (rephrasing or adding whole sentence fragments). Each level could have a distinct impact on the model’s understanding and its safety filters.

In [ ]:
# Grouped analysis - perturbation_level
data.groupby("perturbation_level")["perturbed_response_safety"].value_counts(
    normalize=True
).unstack().mul(100).round(2).reindex(["word", "char", "sntnc"]).T 
#.T.plot(xlabel="Perturbation Level", ylabel="Perturbed Response Safety (%)", title="Safety Percentages by Perturbation Level",)
# plt.legend(title="Category")
# plt.show()

### There higher safety for sntnc level than character and then word

In [ ]:
# Grouped analysis - perturbation_level
data.groupby("perturbation_level")["transition_type"].value_counts(
    normalize=True
).unstack().mul(100).round(2).reindex(["word", "char", "sntnc"]).T #.plot(xlabel="Perturbation Level",ylabel="Perturbed Response Safety (%)",title="Safety Percentages by Perturbation Level")

## 3. Perturbation Type

In [ ]:
# Grouped analysis - perturbation_type
data.groupby("perturbation_type")["perturbed_response_safety"].value_counts(
    normalize=True
).unstack().mul(100).round(2)#T.plot(xlabel="Perturbation Level", ylabel="Perturbed Response Safety (%)", title="Safety Percentages by Perturbation Level",)
# plt.legend(title="Category")
# plt.show()

In [ ]:
# Grouped analysis - perturbation_type
grouped_data = (
    data.groupby("perturbation_type")["transition_type"]
    .value_counts(normalize=True)
    .unstack()
    .mul(100)
    .round(2)
).T.loc[["safe_to_unsafe", "unsafe_to_safe"]].T

grouped_data

In [ ]:
# Create the plot
grouped_data.sort_values(by="unsafe_to_safe", ascending=False).T.plot(
    kind="bar",
    xlabel="Safety Behaviour",
    ylabel="Safety Percetange (%)",
    title="Safety Flip Percentages by Perturbation Type",
    figsize=(15, 8),
    rot=0,
)
plt.legend(title="Safety Behaviour")
plt.show()

In [ ]:
# Filter data to include only 'safe_to_unsafe' and 'unsafe_to_safe'
filtered_data = (
    grouped_data.T.loc[["safe_to_unsafe", "unsafe_to_safe"]]
    .T.sort_values(by="unsafe_to_safe", ascending=False)
    .T
)

# Rename columns for better readability (if applicable)
filtered_data.rename(columns=safety_behaviour_mapping, inplace=True)

# Transpose the data for a cleaner bar plot
filtered_data.T.plot(
    kind="bar",
    xlabel="Perturbation Type",
    ylabel="Safety Percetange (%)",
    title="Safety Regression vs. Improvement by Perturbation Type",
    figsize=(15, 8),
    rot=15,
    color=["#ff7f0e", "#2ca02c"],  # Orange for regression, green for improvement
)

# Customize legend
plt.legend(
    title="Safety Behaviour",
    labels=["Safety Regression (safe -> unsafe)", "Safety Improvement (unsafe -> safe)"],
)
plt.tight_layout()
plt.show()

In [ ]:
grouped_data.loc[["safe_to_unsafe", "unsafe_to_safe"]]